# Driving Question: Can we create a model to identify fossil shark teeth?

# Load the necessary packages

In [ ]:
!pip install torchvision 
!pip install opencv-python
!pip install scikit-learn
!pip install matplotlib
!pip install seaborn
!pip install nbconvert

# Build the model

In [ ]:
# This first bit of code imports the libraries and packages (bundle of different libraries) needed to perform the analysis. 
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:

# Set the device to CPU, so that the model can run on your computer
device = torch.device("cpu")

In [ ]:

from torchvision import transforms

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor()
])

# Load Image Datasets

In [ ]:
# This section loads the datasets of images we will use in our model
import os
import random
from PIL import Image, ImageFilter

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split


# Here we will give names to the folders and images, so that we can keep track of them while we work
base_dir = "Image_Folder"
classes = ["Shark 1", "Shark 2", "Shark 3", "Shark 4", "Shark 5", "Shark 6"]   # -----------MUST ADD SHARK IMAGE DATASETS TO GITHUB AND CHANGE THESE NAMES TO MATCH---------------

# Here we are setting up the folders the model will look at for training
train_val_dirs = [os.path.join(base_dir, f"{cls}_resized_gray") for cls in classes]

# Here we are setting up the folders the model will look at for testing
test_dirs = [os.path.join(base_dir, f"{cls}_test") for cls in classes]


# We will blur a border around the image to reduce discourage the model from looking at the image backgrounds
class RandomBlurBorders:
    """
    Randomly blur a border band around the image to reduce background reliance.
    - edge_ratio: fraction of H/W used as border width (e.g., 0.12 = 12%)
    - p: probability to apply
    - radius: Gaussian blur radius
    Works on PIL.Image; apply before ToTensor().
    """
    def __init__(self, edge_ratio=0.12, p=0.7, radius=3):
        self.edge_ratio = edge_ratio
        self.p = p
        self.radius = radius

    def __call__(self, img: Image.Image) -> Image.Image:
        
        if random.random() > self.p:
            return img

        if img.mode != 'L':
            img = img.convert('L')

        w, h = img.size
        ew, eh = int(w * self.edge_ratio), int(h * self.edge_ratio)
        blurred = img.filter(ImageFilter.GaussianBlur(self.radius))
        out = img.copy()
        
        out.paste(blurred.crop((0, 0, w, eh)), (0, 0))
        out.paste(blurred.crop((0, h - eh, w, h)), (0, h - eh))

        out.paste(blurred.crop((0, 0, ew, h)), (0, 0))
        out.paste(blurred.crop((w - ew, 0, w, h)), (w - ew, 0))

        return out

# Set up all transforms on the images to get them ready for training and validiation using augmentation + normalization

INPUT_SIZE = 128 

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE, scale=(0.80, 1.0), ratio=(0.9, 1.1)),
    transforms.Grayscale(num_output_channels=1),
    RandomBlurBorders(edge_ratio=0.12, p=0.7, radius=3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0.0),
])

eval_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

class CustomImageDataset(Dataset):
    def __init__(self, folders, transform=None):
        self.samples = []
        self.transform = transform
        for label, folder in enumerate(folders):
            if not os.path.isdir(folder):
                continue
            if ".ipynb_checkpoints" in folder:
                continue
            for fname in os.listdir(folder):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp')):
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

# Build datasets (two base datasets for different transforms)
# For splitting, we need consistent indexing; so create a base dataset with no transform just to get samples

base_for_split = CustomImageDataset(train_val_dirs, transform=None)
all_samples = base_for_split.samples
all_labels = [label for _, label in all_samples]

# Stratified split
train_idx, val_idx = train_test_split(
    range(len(all_samples)),
    test_size=0.2,
    stratify=all_labels,
    random_state=42
)

# Now create two datasets with different transforms but same sample list (by reusing base_for_split.samples)
# We wrap a tiny dataset class that uses a shared sample list but distinct transforms

class SharedSamplesDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        if self.transform:
            img = self.transform(img)
        return img, label

train_base = SharedSamplesDataset(all_samples, transform=train_transform)
val_base   = SharedSamplesDataset(all_samples, transform=eval_transform)

# Subsets with the same indices

train_dataset = Subset(train_base, train_idx)
val_dataset   = Subset(val_base,   val_idx)

# Test dataset reads only the _test folders and uses eval_transform

test_dataset  = CustomImageDataset(test_dirs, transform=eval_transform)

# DataLoaders
batch_size = 32

safe_workers = max(0, min(2, (os.cpu_count() or 2) - 1))

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True,
    num_workers=safe_workers, pin_memory=(torch.cuda.is_available()),
    persistent_workers=(safe_workers > 0)
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,
    num_workers=safe_workers, pin_memory=(torch.cuda.is_available()),
    persistent_workers=(safe_workers > 0)
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False,
    num_workers=safe_workers, pin_memory=(torch.cuda.is_available()),
    persistent_workers=(safe_workers > 0)
)

# Check the above setup by printing total images for training, validation and test datasets   -----------------MUST CHECK TOTAL NUMBER OF IMAGES BEING USED HERE AND VERIFY NUMBERS-----------------------------
# Recall we are running this model with 55 images in each of 6 classes, resulting in 330 total training images
# Recall we are also testing it on 5 images in each of 6 classes, resulting in 30 total test images
# Look for these numbers in the output below
print("Total images in training and validation datasets:", len(all_samples))
print("Images in training set:", len(train_dataset))
print("Images in validation set:", len(val_dataset))
print("Images in test set:", len(test_dataset))

# Define the model type

In [ ]:
# Define the CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, 6)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.fc_layer(x)
        return x

In [ ]:
# Initialize Model, Loss, Optimizer
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
# This line of code sets the learning rate of the model
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model for 5 epochs

In [ ]:
# Setting up a training loop to have the model repeat for 5 epochs

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:  
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.to(device), val_labels.to(device)

            val_outputs = model(val_images)
            loss = criterion(val_outputs, val_labels)

            val_loss += loss.item()
            _, val_predicted = torch.max(val_outputs.data, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()

    val_loss /= len(val_loader)
    val_accuracy = 100 * val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f}, Val Accuracy:   {val_accuracy:.2f}%")

# Validate the model

In [ ]:
# Setting up the confusion matrix for validation set
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Set model to evaluation model
model.eval()
    
# Collect predictions and true labels
true_labels = []
pred_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

# Get class names from the datasets. This will label each species with its name
class_names = classes


# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Validation Dataset Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Now, we will ask to see images of the misclassifed teeth
import matplotlib.pyplot as plt

# Set model to evaluation model
model.eval()

misclassified_images = []
misclassified_true_labels = []
misclassified_pred_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        # Find misclassified indices
        for i in range(len(labels)):
            if predicted[i] != labels[i]:
                misclassified_images.append(images[i].cpu())
                misclassified_true_labels.append(labels[i].cpu().item())
                misclassified_pred_labels.append(predicted[i].cpu().item())

# Show up to 10 of the misclassified images
num_to_show = min(10, len(misclassified_images))
plt.figure(figsize=(15, 8))
for idx in range(num_to_show):
    img = misclassified_images[idx]
    true_label = class_names[misclassified_true_labels[idx]]
    pred_label = class_names[misclassified_pred_labels[idx]]

    plt.subplot(2, 5, idx + 1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title(f"True: {true_label}\nPred: {pred_label}", fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

# Test the model

In [ ]:
# Now we will test the model with the unseen test data, 5 teeth per class

model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Average loss across all batches
test_loss /= len(test_loader)
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

In [ ]:
# Confusion matrix for the test dataset results
# Recall that we want to see the results fall along the diagonal line

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

model.eval()
true_labels = []
pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

# Get class names from dataset
class_names = classes

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Test Dataset Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Show images for misclassifed teeth from the test datasets
import matplotlib.pyplot as plt

# Set model to evaluation model
model.eval()

misclassified_images = []
misclassified_true_labels = []
misclassified_pred_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        # Find misclassified indices
        for i in range(len(labels)):
            if predicted[i] != labels[i]:
                misclassified_images.append(images[i].cpu())
                misclassified_true_labels.append(labels[i].cpu().item())
                misclassified_pred_labels.append(predicted[i].cpu().item())

# Show up to 10 of the misclassified images
num_to_show = min(10, len(misclassified_images))
plt.figure(figsize=(15, 8))
for idx in range(num_to_show):
    img = misclassified_images[idx]
    true_label = class_names[misclassified_true_labels[idx]]
    pred_label = class_names[misclassified_pred_labels[idx]]

    plt.subplot(2, 5, idx + 1)
    plt.imshow(img.permute(1, 2, 0))
    plt.title(f"True: {true_label}\nPred: {pred_label}", fontsize=10)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# This function shows the architecture for our SimpleCNN model

print(model)